# **Twitter Sentiment Analysis for Product Feedback**

___


# 1. Exploratory Data Analysis (EDA)

**1.1 Data Loading**

The dataset was loaded successfully using pandas. An initial preview of the data shows multiple columns including an identifier, product name, sentiment label, and tweet text. This step helps in understanding the overall structure and content of the dataset before performing deeper analysis.

In [8]:
import pandas as pd

df = pd.read_csv("/content/twitter_training.csv")


**1.2 Dataset Overview**

An initial inspection of the dataset provides an overview of its structure, including identifiers, product/entity names, sentiment labels, and raw tweet text. This helps in understanding the nature of the data before further analysis.


In [7]:
df.head()

,2401,Borderlands,Positive,"im getting on borderlands and i will murder you all ,"
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...


**1.3 Dataset Structure and Information**

The dataset contains 74,681 entries and 4 columns.
The columns include one integer identifier and three object-type columns representing product name, sentiment label, and tweet text.
This confirms that the dataset is primarily textual in nature and suitable for Natural Language Processing tasks.

In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 74681 entries, 0 to 74680
Data columns (total 4 columns):
 #   Column                                                 Non-Null Count  Dtype 
---  ------                                                 --------------  ----- 
 0   2401                                                   74681 non-null  int64 
 1   Borderlands                                            74681 non-null  object
 2   Positive                                               74681 non-null  object
 3   im getting on borderlands and i will murder you all ,  73995 non-null  object
dtypes: int64(1), object(3)
memory usage: 2.3+ MB


The dataset consists of the following columns:



*   An ID column representing tweet or record identifiers
*   A product/entity column (e.g., Borderlands)
*   A sentiment column (e.g., Positive)
*   A tweet text column containing raw user-generated content

The sentiment column will be used as the target variable for supervised learning.

**1.4 Missing Value Analysis**

Missing value analysis was performed across all columns to identify incomplete records.

The results show that the identifier, product, and sentiment columns contain no missing values.

However, the tweet text column contains 686 null entries, which indicates the presence of incomplete textual data.

Since text content is essential for Natural Language Processing tasks, these records will be removed during preprocessing to ensure data quality and prevent errors during feature extraction.

In [6]:
df.isnull().sum()

,0
2401,0
Borderlands,0
Positive,0
"im getting on borderlands and i will murder you all ,",686


**1.5 Data Quality Observations**

1.5.1 Qualitative Inspection of Tweets

A random sample of tweets was examined to assess text quality. The sampled tweets contain informal language, slang, offensive expressions, and inconsistent grammar, which are characteristic of social media data. This confirms that raw tweet text is noisy and requires thorough cleaning and normalization before feature extraction.

In [9]:
df['im getting on borderlands and i will murder you all ,'].sample(5, random_state=42).values

array(["went to go in george's room to find his door was locked? climbed up side of house to look through his window expecting to see a regrettable girl or something but no...he was playing...Fortnite????",
       'Yo this looks LIT! Team:GO/Overwatch combo',
       'Pay attention executive administrators. While your stores are taking delivery orders for pallets of bricks to downtown then you were being used in helping the destruction of private property.',
       'Guy looked at me and says my name was put on the throw list lmaoooo get a fuck to watch you weird ass dude',
       'one'], dtype=object)

1.5.2 Tweet Length Analysis

Tweet length analysis shows significant variability in text size, ranging from very short tweets to long textual content. This wide distribution indicates the need for normalization and vectorization techniques such as TF-IDF, which handle varying text lengths effectiv

In [10]:
df['tweet_length'] = df['im getting on borderlands and i will murder you all ,'].astype(str).apply(len)

df['tweet_length'].describe()


,tweet_length
count,74681.000000
mean,107.812697
std,79.799121
min,1.000000
25%,45.000000
50%,90.000000
75%,152.000000
max,957.000000


1.5.3 Duplicate Tweet Analysis

Duplicate analysis revealed approximately 5,190 duplicate tweets in the dataset. Since duplicate entries can bias the learning process and over-represent certain opinions, duplicate tweets will be removed during preprocessing.

In [14]:
df['im getting on borderlands and i will murder you all ,'].duplicated().sum()


np.int64(5190)

1.5.4 Noise Pattern Detection

A substantial number of tweets contain noise patterns such as URLs, hashtags, or special characters. These elements do not contribute to sentiment understanding and will be removed during text cleaning.

In [15]:
df['im getting on borderlands and i will murder you all ,'].str.contains(
    r"http|@|#", regex=True, na=False
).sum()


np.int64(20035)

**1.6 EDA Summary**

The dataset contains labeled Twitter data suitable for sentiment analysis. EDA revealed noisy and informal text, variable tweet lengths, duplicate entries, and a small number of missing values. These findings justified the need for thorough text preprocessing before model training.

<hr style="width:100%;">





# 2. Data Preprocessing

**2.1 Column Renaming for Clarity**

Column names were renamed to meaningful and concise identifiers to improve readability and maintainability of the preprocessing and modeling pipeline.

In [19]:
df = df.rename(columns={
    df.columns[0]: 'id',
    df.columns[1]: 'product',
    df.columns[2]: 'sentiment',
    df.columns[3]: 'tweet'
})
df.head()


,id,product,sentiment,tweet,tweet_length
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...,51
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...,50
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...,51
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...,57
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...,53


**2.2 Handling Missing Values**

Rows with missing tweet text were removed since textual content is mandatory for sentiment analysis and such records do not contribute useful information.

In [20]:
df = df.dropna(subset=['tweet'])

**2.3 Removing Duplicate Tweets**

Duplicate tweets were removed to prevent bias caused by repeated content and to ensure balanced learning during model training.

In [21]:
df = df.drop_duplicates(subset=['tweet'])

**2.4 Text Cleaning (Noise Removal)**

Text cleaning was performed to remove noise such as URLs, mentions, hashtags, punctuation, and numeric characters. All text was converted to lowercase to ensure consistency during feature extraction.

In [22]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)   # URLs
    text = re.sub(r"@\w+", "", text)             # mentions
    text = re.sub(r"#\w+", "", text)             # hashtags
    text = re.sub(r"[^a-z\s]", "", text)         # punctuation & numbers
    text = re.sub(r"\s+", " ", text).strip()     # extra spaces
    return text

df['clean_tweet'] = df['tweet'].apply(clean_text)
df[['tweet', 'clean_tweet']].head()


,tweet,clean_tweet
0,I am coming to the borders and I will kill you...,i am coming to the borders and i will kill you...
1,im getting on borderlands and i will kill you ...,im getting on borderlands and i will kill you all
2,im coming on borderlands and i will murder you...,im coming on borderlands and i will murder you...
3,im getting on borderlands 2 and i will murder ...,im getting on borderlands and i will murder yo...
4,im getting into borderlands and i can murder y...,im getting into borderlands and i can murder y...


**2.5 Stopword Removal**

Common stopwords were removed to reduce noise and focus on sentiment-bearing terms that contribute meaningfully to classification.

In [26]:
import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

stop_words = set(stopwords.words('english'))

df['clean_tweet'] = df['clean_tweet'].apply(
    lambda x: " ".join([word for word in x.split() if word not in stop_words])
)


[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


**2.6 Lemmatization**

Lemmatization was applied to convert words to their base form while preserving semantic meaning, improving generalization during model training.

In [29]:
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')

lemmatizer = WordNetLemmatizer()

df['clean_tweet'] = df['clean_tweet'].apply(
    lambda x: " ".join([lemmatizer.lemmatize(word) for word in x.split()])
)

df.head()

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


,id,product,sentiment,tweet,tweet_length,clean_tweet
0,2401,Borderlands,Positive,I am coming to the borders and I will kill you...,51,coming border kill
1,2401,Borderlands,Positive,im getting on borderlands and i will kill you ...,50,im getting borderland kill
2,2401,Borderlands,Positive,im coming on borderlands and i will murder you...,51,im coming borderland murder
3,2401,Borderlands,Positive,im getting on borderlands 2 and i will murder ...,57,im getting borderland murder
4,2401,Borderlands,Positive,im getting into borderlands and i can murder y...,53,im getting borderland murder


### 2.7 Preprocessing Summary

- Renamed columns for clarity and consistency  
- Removed rows with missing tweet text  
- Eliminated duplicate tweets to reduce bias  
- Cleaned raw text by removing noise and normalizing case  
- Removed stopwords to focus on informative terms  
- Applied lemmatization to preserve semantic meaning  

The processed text is now suitable for feature extraction and model training.


___

# 3. Feature Engineering and Model Preparation

**3.1 Train–Test Split**

The dataset was divided into training and testing sets using an 80–20 split. Stratified sampling was applied to preserve the original sentiment distribution across both sets.

In [30]:
from sklearn.model_selection import train_test_split

X = df['clean_tweet']
y = df['sentiment']

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


**3.2 Text Vectorization using TF-IDF**

TF-IDF vectorization was applied to transform textual data into numerical features. Unigrams and bigrams were used to capture contextual information, while limiting the feature space to the most informative terms.


In [31]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2)
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)


**3.3 Feature Matrix Shape Inspection**

The resulting feature matrices confirm successful vectorization and appropriate dimensionality for model training and evaluation.

In [32]:
X_train_tfidf.shape, X_test_tfidf.shape


((55592, 5000), (13898, 5000))

**3.4 Model Selection**

Why Logistic Regression?


*  Fast
*  Interpretable
*   Excellent for TF-IDF features


**3.5 Model Training**

Logistic Regression was selected as the baseline classifier due to its effectiveness with high-dimensional sparse text features generated by TF-IDF.

In [33]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train_tfidf, y_train)


LogisticRegression(max_iter=1000)

**3.6 Model Prediction**



In [34]:
y_pred = model.predict(X_test_tfidf)


**3.7 Model Evaluation**

Model performance was evaluated using precision, recall, and F1-score to account for potential class imbalance in sentiment categories.

In [35]:
from sklearn.metrics import classification_report, confusion_matrix

print(classification_report(y_test, y_pred))
confusion_matrix(y_test, y_pred)


              precision    recall  f1-score   support

  Irrelevant       0.66      0.50      0.57      2443
    Negative       0.71      0.78      0.75      4233
     Neutral       0.65      0.63      0.64      3409
    Positive       0.68      0.73      0.70      3813

    accuracy                           0.68     13898
   macro avg       0.67      0.66      0.66     13898
weighted avg       0.68      0.68      0.68     13898



array([[1219,  400,  359,  465],
       [ 176, 3322,  401,  334],
       [ 233,  538, 2139,  499],
       [ 232,  406,  407, 2768]])

### 3.8 Feature Engineering Summary

- Cleaned tweet text was split into training and testing sets  
- TF-IDF vectorization converted text into numerical features  
- Logistic Regression was trained as a baseline classifier  
- Model performance was evaluated using standard classification metrics  


___


# 4. End-to-End NLP Pipeline & Model Persistence

**4.1 Why Use a Pipeline?**

Instead of doing:

clean text → TF-IDF → model (manually)



We bundle everything into one object:

raw cleaned text → TF-IDF → classifier → prediction


**4.2 Build the sklearn Pipeline**

In [36]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        max_features=5000,
        ngram_range=(1, 2)
    )),
    ('classifier', LogisticRegression(max_iter=1000))
])


**4.3 Train the Pipeline**

In [37]:
pipeline.fit(X_train, y_train)


Pipeline(steps=[('tfidf',
                 TfidfVectorizer(max_features=5000, ngram_range=(1, 2))),
                ('classifier', LogisticRegression(max_iter=1000))])

**4.4 Evaluate the Pipeline**

In [38]:
from sklearn.metrics import classification_report

y_pred_pipeline = pipeline.predict(X_test)
print(classification_report(y_test, y_pred_pipeline))


              precision    recall  f1-score   support

  Irrelevant       0.66      0.50      0.57      2443
    Negative       0.71      0.78      0.75      4233
     Neutral       0.65      0.63      0.64      3409
    Positive       0.68      0.73      0.70      3813

    accuracy                           0.68     13898
   macro avg       0.67      0.66      0.66     13898
weighted avg       0.68      0.68      0.68     13898



The classification report indicates balanced performance across sentiment classes. The model performs best on negative and positive sentiments, which typically contain stronger lexical cues. Neutral and irrelevant classes show comparatively lower scores due to overlapping language patterns and ambiguity inherent in social media text. Overall, the results demonstrate that the model generalizes reasonably well across multiple sentiment categories.

**4.5 Save the Pipeline**

In [39]:
import pickle

with open("sentiment_pipeline.pkl", "wb") as file:
    pickle.dump(pipeline, file)


**4.6 Load & Test Saved Pipeline**

In [40]:
with open("sentiment_pipeline.pkl", "rb") as file:
    loaded_pipeline = pickle.load(file)

loaded_pipeline.predict(["this product is absolutely amazing"])


array(['Positive'], dtype=object)

### 4.7 Pipeline Summary

- An end-to-end NLP pipeline was constructed using TF-IDF vectorization and Logistic Regression  
- The pipeline ensures consistent preprocessing and prediction  
- The trained pipeline was serialized using pickle for deployment  
- The saved model can be directly integrated into web applications for real-time inference  
